# GWAD reproduction on Colab

CVPR 2025 *Mind the Gap: Detecting Black-box Adversarial Attacks in the Making through Query Update Analysis*.

1. `Runtime > Change runtime type > T4 GPU`
2. Run cells top to bottom. Weights / CIFAR-10 / results are cached in Google Drive (`MyDrive/GWAD`), so a disconnect does not lose them.
3. Colab free tier: max ~12h per session, so keep `N_IMAGES` small (HSJA ~1-2 min per image).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/GWAD'
!mkdir -p {DRIVE_DIR}/model {DRIVE_DIR}/data {DRIVE_DIR}/stats

In [ ]:
REPO_URL = 'https://github.com/Chu-c-git/GWAD_reproduction.git'
%cd /content
!git clone -q {REPO_URL} GWAD || git -C GWAD pull -q
%cd /content/GWAD
# torch / torchvision / numpy / scikit-image / opencv / matplotlib / tqdm are preinstalled on Colab
!pip install -q gdown truststore

## Target model weights + CIFAR-10 (cached in Drive)

In [ ]:
import os
if not os.path.exists(f'{DRIVE_DIR}/model/resnet18.pt'):
    !python scripts/download_weights.py
    !cp model/cifar10/resnet18.pt {DRIVE_DIR}/model/
!mkdir -p model/cifar10 && cp {DRIVE_DIR}/model/resnet18.pt model/cifar10/

# code reads CIFAR-10 from ./data -> point it at Drive so the download happens once
!rm -rf data && ln -s {DRIVE_DIR}/data data
!python -c "from torchvision import datasets; datasets.CIFAR10('./data', train=False, download=True)"
!ls -la model/cifar10 data/

## Run

- `ATTACK`: `hsja`, `nes`, `sign-flip`
- `SCENARIO`: `benign`, `standard`, `batch`
- `N_IMAGES`: benign = number of benign queries, attack = number of attacked images (batch scenario first uses 2000 images as benign pool, these are not counted)

In [ ]:
ATTACK = 'hsja'
SCENARIO = 'standard'
N_IMAGES = 5

!python demo.py --data cifar10 --attack {ATTACK} --scenario {SCENARIO} --n_images {N_IMAGES} 2>&1 | tee {DRIVE_DIR}/stats/log_{ATTACK}_{SCENARIO}.txt

In [ ]:
# save DS histograms / DS distributions written to stats/
!cp -r stats/hist stats/dist {DRIVE_DIR}/stats/
!ls {DRIVE_DIR}/stats/hist

## Plot mean delta-similarity histogram (2001 bins over [-1, 1], not the 201-bin HoDS; Python port of `stats/show_stats.m`)

In [ ]:
import glob
import numpy as np
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
for f in sorted(glob.glob('stats/hist/*.txt')):
    h = np.loadtxt(f, delimiter=',')
    plt.plot(np.linspace(-1, 1, len(h)), h, label=os.path.basename(f)[:-4])
plt.xlabel('delta similarity (cosine)')
plt.ylabel('mean count')
plt.legend()
plt.show()